# Transfer Learning with Pre-Trained CNN Models in PyTorch: Feature Extraction vs. Fine-Tuning (VGG16 & ResNet50 on CIFAR-10)


### Goals (beginner-level)
- See **why** we reuse ImageNet models for small images (CIFAR-10).
- Load ready-made **VGG16** and **ResNet50** from `torchvision` and change the **last layer** to 10 classes.
- Run a simple **training loop**: forward → loss → backward → optimizer step.
- Try **feature extraction** (train head only) vs **fine-tuning** (train head + last blocks).
- Complete **three exercises**: each has a **coding** part plus **written questions** (minimum **100 words** each).

**Total Marks:** **100** (see each exercise for breakdown)

## Contents

1. [Title & Overview](#lab-4-transfer-learning-with-pre-trained-cnn-models-in-pytorch)
2. [Detailed Tutorial Section](#tutorial-transfer-learning-end-to-end)
3. [Lab Exercises (3 exercises, 100 marks)](#lab-exercises-100-marks)
4. [Conclusion & Submission](#conclusion--submission)


## Reading this notebook

- Run cells **from in order** (Shift+Enter). Wait for each cell to finish before the next.
- **GPU:** If you use Google Colab, turn on **Runtime → Change runtime type → GPU**. Training is much faster.
- **Stuck?** Read the **comments** in code cells — they explain what each line does.
- The **tutorial** is self-contained. The **exercises** ask you to fill in `# TODO` lines using the same patterns.


### PyTorch reminders (very short)

| Idea | What it means |
|------|----------------|
| `tensor` | A multi-dimensional array (like NumPy), often on GPU. |
| `net.to(device)` | Move the network to CPU or GPU. |
| `net.train()` / `net.eval()` | Training mode vs test mode (affects dropout/batch norm). |
| `requires_grad` | If `False`, PyTorch will **not** update those weights (we **freeze** layers). |
| `DataLoader` | Gives you **batches** of images and labels in a loop. |
| `CrossEntropyLoss` | Standard loss for multi-class classification. |

You do **not** need to memorize this — refer back anytime.


---
<a id="tutorial-transfer-learning-end-to-end"></a>
# Detailed Tutorial Section (~60% of notebook)

This section is **fully working**. Run it **in order** like a script. When it runs without errors, you are ready for the exercises below.


## What is Transfer Learning?

Someone already trained a big CNN on **ImageNet** (millions of photos, 1000 classes). That net learned **general** visual skills (edges, shapes, textures).

**Transfer learning** means: take that trained net, **swap the last layer** for our 10 CIFAR-10 classes, and train only what we need. We save time and often get **better accuracy** than training from random weights.


## Pre-trained models we use (names only)

**VGG16** — Older, simple design: many convolution layers in a row, then fully connected layers. Easy to understand.

**ResNet50** — Newer, **deeper** network with **skip connections** (shortcuts that help training). In code it has parts called `layer1` … `layer4`. For fine-tuning we often train **`layer4` + the final `fc` layer**.


## Feature Extraction vs. Fine-Tuning

| Aspect | Feature Extraction | Fine-Tuning |
|--------|-------------------|-------------|
| **What moves?** | Only the new classifier head | Head + selected/top backbone layers |
| **Pros** | Fast, less overfitting risk with small data | Higher accuracy when data is sufficient |
| **Cons** | May underfit if domain differs | More compute; needs smaller LR on backbone |
| **When to use** | Small dataset, similar to ImageNet | More data or domain gap |
| **LR strategy** | Moderate LR on head (e.g., 1e-3) | Smaller LR on backbone (e.g., 1e-4), often larger on head |
| **Compute** | Lower | Higher |


## Step 1 — Imports and device

`device` will be `"cuda"` (GPU) if available, otherwise `"cpu"`. Always send **models** and **data batches** to the same device with `.to(device)`.


In [ ]:
# --- Standard libraries ---
import os
import time
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm  # progress bar for training loops

# --- PyTorch core ---
import torch
import torch.nn as nn  # layers like Linear, Conv2d
import torch.optim as optim  # optimizers like SGD, Adam
from torch.utils.data import DataLoader, Subset  # load data in batches

# --- torchvision: datasets + pre-trained models ---
import torchvision
import torchvision.transforms as transforms  # image preprocessing
from torchvision import models  # VGG, ResNet, etc.
from torchvision.datasets import CIFAR10

# Reproducibility (same random numbers each run — good for teaching / debugging)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found — training will be slower; consider Google Colab with GPU.")


### 📦 Data & Dependency Setup
Since GitHub only stores the notebook file and not your local Colab files, run this cell to ensure all necessary data is downloaded into the current environment.

In [ ]:
# Example: Downloading a dataset if it doesn't exist
import os

# If you have specific files in sample_data you need, you can move/link them,
# but typically you would download your project files here:
# !wget https://raw.githubusercontent.com/your-username/your-repo/main/your-data-file.csv

if not os.path.exists('./data'):
    os.makedirs('./data', exist_ok=True)
    print("Data directory created. The CIFAR-10 download in Step 2 will use this path.")
else:
    print("Environment ready.")

## Step 2 — Load CIFAR-10 and resize images

CIFAR-10 images are **32×32**. Pre-trained ImageNet models expect **224×224**, so we **resize**.

We also **normalize** with ImageNet mean/std — this matches how the pre-trained weights were trained.


In [ ]:
# Numbers ImageNet models expect (do not change these when using ImageNet weights)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# transforms.Compose chains steps left → right
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((224, 224)),  # make small images bigger for the CNN
        transforms.ToTensor(),  # convert PIL image to tensor, scale 0–1
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "test": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
}

batch_size = 32
# Use num_workers=0 if you see errors on Windows; 2 is OK on many Linux/Mac setups
num_workers = 0

train_set = CIFAR10(root="./data", train=True, download=True, transform=data_transforms["train"])
test_set = CIFAR10(root="./data", train=False, download=True, transform=data_transforms["test"])

train_loader = DataLoader(
    train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available()
)

classes = train_set.classes  # list of 10 class names as strings
print("Class names:", classes)
print("Number of training batches:", len(train_loader), "| test batches:", len(test_loader))


### See the dataset (helps you understand the task)

CIFAR-10 is **10 classes** of small photos (airplane, car, bird, …). Our net will output **one label per image**.

Below we plot images **without** ImageNet normalization so colors look normal on screen. Training still uses the normalized tensors from the cell above.


In [ ]:
# Visualization-only transforms: resize + tensor in [0, 1] (no Normalize — easier for imshow)
viz_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
viz_train = CIFAR10(root="./data", train=True, download=True, transform=viz_transform)

# --- Figure 1: one training image per class (so you see all 10 labels) ---
one_per_class = {}
for idx in range(len(viz_train)):
    img_t, label = viz_train[idx]
    if label not in one_per_class:
        one_per_class[label] = img_t
    if len(one_per_class) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    img = one_per_class[i]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(classes[i], fontsize=11)
    ax.axis("off")
plt.suptitle("CIFAR-10: one example per class (training set, resized to 224×224)", fontsize=13)
plt.tight_layout()
plt.show()

# --- Figure 2: random mini-batch — shows diversity within classes ---
viz_loader = DataLoader(viz_train, batch_size=16, shuffle=True, num_workers=num_workers)
imgs_batch, labels_batch = next(iter(viz_loader))
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs_batch[i].permute(1, 2, 0).numpy())
    ax.set_title(classes[labels_batch[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("16 random training images (with class names)", fontsize=13)
plt.tight_layout()
plt.show()

print("Task: given a 224×224 image like above, predict 1 of 10 class names.")


## Step 3 — Build models with 10 output classes

We download weights with `weights="IMAGENET1K_V1"`. The last layer was trained for **1000** ImageNet classes; we replace it with **10** for CIFAR-10.

- **VGG16:** the last linear layer is `net.classifier[6]`.
- **ResNet50:** the last layer is `net.fc`.
- **Freeze** = set `requires_grad = False` so those weights are **not** updated during training.


In [ ]:
def get_vgg16_feature_extractor(num_classes=10):
    # Load VGG16; freeze convolutional part; only train the small classifier at the end.
    net = models.vgg16(weights="IMAGENET1K_V1")
    for p in net.features.parameters():
        p.requires_grad = False  # freeze all convolution layers
    in_f = net.classifier[6].in_features  # input size of last linear layer
    net.classifier[6] = nn.Linear(in_f, num_classes)  # 10 outputs for CIFAR-10
    return net


def get_resnet50_for_cifar(num_classes=10):
    # Load ResNet50 and replace the final fully connected layer only.
    net = models.resnet50(weights="IMAGENET1K_V1")
    in_f = net.fc.in_features
    net.fc = nn.Linear(in_f, num_classes)
    return net


def count_trainable_parameters(net):
    # How many weights will be updated (smaller = faster, less memory).
    return sum(p.numel() for p in net.parameters() if p.requires_grad)


def trainable_params(net):
    # Use this list in optim.SGD(...) — only parameters with requires_grad True.
    return [p for p in net.parameters() if p.requires_grad]


## Step 4 — One training loop you can reuse

`evaluate_model` = measure loss and accuracy **without** updating weights (`torch.no_grad()`).

`train_model` = for each epoch: loop over training batches, **forward → loss → backward → optimizer.step()**, then measure on the test set. This pattern appears in almost every PyTorch classification lab.


In [ ]:
def evaluate_model(net, loader, criterion, device):
    # Run on all batches; return average loss and accuracy (no weight updates).
    net.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():  # do not compute gradients — faster and less memory
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)  # class with highest score
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total


def train_model(net, train_loader, test_loader, criterion, optimizer, device, num_epochs, scheduler=None, model_name="net"):
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    since = time.time()
    best_acc = 0.0
    best_wts = copy.deepcopy(net.state_dict())

    for epoch in range(num_epochs):
        net.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f"{model_name} Epoch {epoch+1}/{num_epochs}")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()  # clear old gradients
            outputs = net(inputs)  # forward pass
            loss = criterion(outputs, labels)  # compare predictions to true labels
            loss.backward()  # compute gradients
            optimizer.step()  # update weights

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix(loss=loss.item())

        epoch_loss = running_loss / total
        epoch_acc = correct / total

        test_loss, test_acc = evaluate_model(net, test_loader, criterion, device)
        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(test_loss)
            else:
                scheduler.step()

        history["train_loss"].append(epoch_loss)
        history["train_acc"].append(epoch_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} Acc={epoch_acc:.4f} | Test Loss={test_loss:.4f} Acc={test_acc:.4f}")

        if test_acc > best_acc:
            best_acc = test_acc
            best_wts = copy.deepcopy(net.state_dict())

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed:.1f}s | Best test acc: {best_acc:.4f}")
    net.load_state_dict(best_wts)
    return net, history, time_elapsed, best_acc


## Step 5 — Example A: Feature extraction (VGG16, train the head only)

We only train the **new last layer**. Learning rate **0.001** is a common starting point.

*Tip: we use **3 epochs** below so the tutorial runs quickly on CPU. You can change `TUTORIAL_EPOCHS` to **5** for slightly better curves.*


In [ ]:
criterion = nn.CrossEntropyLoss()  # expects raw logits and class indices as labels

TUTORIAL_EPOCHS = 3  # increase to 5 if you have time / GPU

# Feature extraction on VGG16
vgg_fe = get_vgg16_feature_extractor(10).to(device)
print("How many weights will train? (VGG16 feature extraction):", count_trainable_parameters(vgg_fe))

opt_fe = optim.SGD(trainable_params(vgg_fe), lr=0.001, momentum=0.9)

vgg_fe, hist_fe, t_fe, best_fe = train_model(
    vgg_fe, train_loader, test_loader, criterion, opt_fe, device,
    num_epochs=TUTORIAL_EPOCHS, scheduler=None, model_name="VGG16_FE"
)


## Step 6 — Example B: Fine-tuning (ResNet50, train `layer4` + head)

We **unfreeze** the last big block (`layer4`) and the final layer `fc`. Earlier layers stay frozen.

We use a **smaller** learning rate for `layer4` than for `fc` so we do not destroy good pre-trained features.


In [ ]:
def setup_resnet50_finetune(num_classes=10):
    net = models.resnet50(weights="IMAGENET1K_V1")
    for p in net.parameters():
        p.requires_grad = False  # start: freeze everything
    for p in net.layer4.parameters():
        p.requires_grad = True  # allow last conv block to train
    in_f = net.fc.in_features
    net.fc = nn.Linear(in_f, num_classes)
    for p in net.fc.parameters():
        p.requires_grad = True
    return net


rn_ft = setup_resnet50_finetune(10).to(device)
print("How many weights will train? (ResNet50 fine-tune):", count_trainable_parameters(rn_ft))

# Two learning rates: smaller for conv layers, a bit larger for the classifier head
params_ft = [
    {"params": rn_ft.layer4.parameters(), "lr": 1e-4},
    {"params": rn_ft.fc.parameters(), "lr": 1e-3},
]
opt_ft = optim.SGD(params_ft, momentum=0.9)

rn_ft, hist_ft, t_ft, best_ft = train_model(
    rn_ft, train_loader, test_loader, criterion, opt_ft, device,
    num_epochs=TUTORIAL_EPOCHS, scheduler=None, model_name="ResNet50_FT"
)


## Training Curves (Tutorial)


In [ ]:
def plot_history(hist_dict, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(hist_dict["train_loss"], label="train")
    ax[0].plot(hist_dict["test_loss"], label="test")
    ax[0].set_title("Loss")
    ax[0].legend()
    ax[1].plot(hist_dict["train_acc"], label="train")
    ax[1].plot(hist_dict["test_acc"], label="test")
    ax[1].set_title("Accuracy")
    ax[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_history(hist_fe, "VGG16 Feature Extraction")
plot_history(hist_ft, "ResNet50 Fine-Tuning (layer4 + fc)")


## Final Comparison (Tutorial)


In [ ]:
import pandas as pd

# Re-evaluate for clean numbers
_, acc_fe = evaluate_model(vgg_fe, test_loader, criterion, device)
_, acc_ft = evaluate_model(rn_ft, test_loader, criterion, device)

summary = pd.DataFrame([
    {
        "Method": "VGG16 Feature Extraction",
        "Test Acc": acc_fe,
        "Train Time (s)": t_fe,
        "Trainable Params": count_trainable_parameters(get_vgg16_feature_extractor(10)),
    },
    {
        "Method": "ResNet50 Fine-Tune (layer4)",
        "Test Acc": acc_ft,
        "Train Time (s)": t_ft,
        "Trainable Params": count_trainable_parameters(setup_resnet50_finetune(10)),
    },
])
print(summary.to_string(index=False))


### Key Takeaways (Tutorial)
- **Feature extraction** = only the new last layers learn; quick and stable.
- **Fine-tuning** = last blocks + head learn; often better accuracy, needs smaller LR on early layers.
- Use **ImageNet** resize + normalize when you use **ImageNet** weights.


---
<a id="lab-exercises-100-marks"></a>
# Exercises (100 marks total)

There are **three exercises**. Each exercise has:
- **Code:** complete the `# TODO` cells (copy patterns from the tutorial).
- **Questions**. Each answer must be at least **100 words**. State your **word count** at the end of each answer.

| Exercise | Topic (Part A) | Approx. marks |
|----------|----------------|---------------|
| **1** | Data augmentation + visualization | **34** |
| **2** | VGG16 feature extraction | **33** |
| **3** | ResNet50 fine-tuning + LR scheduler + save/load | **33** |

**How to work:** Run the **tutorial** first. Use fewer epochs (e.g. 3) while debugging.


## Exercise 1 — Data augmentation & visualization (34 marks)

 Add `RandomHorizontalFlip`, `RandomRotation`, and `ColorJitter` to the training pipeline; build a `DataLoader`; display **10** augmented images in a grid with class labels (denormalize for display).

Answer the **three** questions below (each **≥ 100 words**).

**Tip:** Create a **second** `CIFAR10` dataset with your new transform; same `root='./data'` as the tutorial.


In [ ]:
# === YOUR CODE STARTS HERE ===
train_transform_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

aug_train_set = CIFAR10(root="./data", train=True, download=True, transform=train_transform_aug)
aug_train_loader = DataLoader(aug_train_set, batch_size=32, shuffle=True, num_workers=num_workers, pin_memory=torch.cuda.is_available())

def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return tensor * std + mean

images, labels = next(iter(aug_train_loader))

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for i in range(10):
    ax = axes.flat[i]
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(classes[labels[i]])
    ax.axis("off")
plt.suptitle("Exercise 1: 10 Augmented Training Images", fontsize=16)
plt.tight_layout()
plt.show()
# === YOUR CODE ENDS HERE ===


### Exercise 1 — Question 1 (≥100 words)

**Prompt:** Explain what **data augmentation** is and why it is commonly used when training (or fine-tuning) CNNs on datasets like CIFAR-10. Discuss how augmentation relates to **overfitting** and **generalization**, and give at least one reason it is still useful even when starting from **pre-trained** ImageNet weights.

**Answer:**
Data augmentation is a technique used to artificially expand the size and diversity of a training dataset by applying various random transformations to the existing images, such as horizontal flipping, rotations, cropping, and color jittering. In the context of datasets like CIFAR-10, where images are relatively low-resolution and limited in number, augmentation is crucial because it prevents the Convolutional Neural Network (CNN) from simply memorizing the training samples. By presenting slightly different versions of the same image in every epoch, the net is forced to learn more robust and invariant features, such as the general shape of an airplane regardless of its orientation or the specific lighting conditions in the photo.

This process is directly related to reducing overfitting and improving generalization. Overfitting occurs when a net performs exceptionally well on training data but fails on unseen test data because it has learned noise rather than patterns. Augmentation acts as a regularizer, ensuring the net generalizes better by simulating a much larger and more varied dataset. Even when starting from pre-trained ImageNet weights, data augmentation remains highly useful. While the backbone already understands low-level features like edges, the target dataset (CIFAR-10) may have different distribution characteristics. Augmentation helps the net adapt those pre-existing features to the specific spatial and color variances of the new task, ensuring that the 'fine-tuning' process doesn't result in the net becoming too rigid or biased toward the specific training examples provided.

## Exercise 2 — VGG16 feature extraction (33 marks)

Train a **VGG16** net in **feature-extraction** mode (`get_vgg16_feature_extractor`), **8 epochs**, LR **0.001**, plot **training curves**, and report **test accuracy**.


Use variable names **`vgg_ex2`** and **`hist_ex2`** for your net and history.


In [ ]:
# === YOUR CODE STARTS HERE ===
# Copy tutorial Step 5, with new names: vgg_ex2, hist_ex2
EX2_EPOCHS = 8

# Initialize net
vgg_ex2 = get_vgg16_feature_extractor(num_classes=10).to(device)
print(f"Trainable parameters for VGG16 FE: {count_trainable_parameters(vgg_ex2):,}")

# Setup optimizer - only training the head
opt = optim.SGD(trainable_params(vgg_ex2), lr=0.001, momentum=0.9)

# Train the net
vgg_ex2, hist_ex2, t_ex2, best_acc_ex2 = train_model(
    vgg_ex2,
    train_loader,
    test_loader,
    criterion,
    opt,
    device,
    num_epochs=EX2_EPOCHS,
    scheduler=None,
    model_name="Ex2_VGG16"
)

# Results and Visualization
plot_history(hist_ex2, "Exercise 2: VGG16 Feature Extraction")

test_loss, test_acc = evaluate_model(vgg_ex2, test_loader, criterion, device)
print(f"Final Test Accuracy: {test_acc:.4f}")

# === YOUR CODE ENDS HERE ===


### Exercise 2 — Question 1 (≥100 words)

**Prompt:** Explain **feature extraction** in your own words: what parts of VGG16 are **frozen**, what part is **trained**, and **why** we usually freeze early layers when using ImageNet pre-training. Connect this to the idea of **low-level** vs **high-level** features.

**Answer:**
Feature extraction is a transfer learning strategy where we utilize the robust visual representations learned by a net on a large dataset like ImageNet and apply them to a new task. In our VGG16 implementation, the 'features' part of the network—which consists of all the convolutional and pooling layers—is frozen by setting `requires_grad = False`. Only the newly added classifier head (the final linear layer) is trained. This approach is highly effective because early convolutional layers in a CNN learn low-level features, such as simple edges, corners, and basic textures, which are universal across almost all images.

As we move deeper into the network, the layers begin to recognize high-level features like specific shapes or complex objects. By freezing the early and middle layers, we preserve these general visual skills that the net already mastered. We only train the final layer to map these sophisticated high-level features to our specific set of 10 CIFAR-10 classes. This significantly reduces the computational cost and the risk of overfitting, as we are only optimizing a small number of parameters compared to the millions contained in the entire VGG16 backbone. This is particularly useful when the new dataset is small or when we want a fast, stable training process.

Word count: 212

## Exercise 3 — ResNet50 fine-tuning, scheduler & checkpoint (33 marks)

Fine-tune **ResNet50** (`setup_resnet50_finetune`): **unfreeze `layer4` + `fc`**, use **two learning rates**, add **`ReduceLROnPlateau`**, train **8 epochs** (use **3** while debugging), plot curves, report test accuracy and final learning rates.

**Also:** Save `ex3_model.state_dict()` to a file, load into a **fresh** net, and **verify** test accuracy matches (see TODOs).

Use **`ex3_model`** and **`hist_ex3`** for your trained net and history.


In [ ]:
# === YOUR CODE STARTS HERE ===
# Copy tutorial Step 6; add ReduceLROnPlateau; use ex3_model, hist_ex3
EX3_EPOCHS = 8

# Initialize net with fine-tuning setup
ex3_model = setup_resnet50_finetune(10).to(device)
print(f"Trainable parameters for ResNet50 FT: {count_trainable_parameters(ex3_model):,}")

# Optimizer with differential learning rates: lower for backbone, higher for head
opt_ex3 = optim.SGD([
    {"params": ex3_model.layer4.parameters(), "lr": 1e-4},
    {"params": ex3_model.fc.parameters(), "lr": 1e-3}
], momentum=0.9)

# Scheduler that reduces LR when validation loss stops improving
sched_ex3 = optim.lr_scheduler.ReduceLROnPlateau(opt_ex3, mode="min", factor=0.5, patience=2)

# Train the net
ex3_model, hist_ex3, t_ex3, best_acc_ex3 = train_model(
    ex3_model,
    train_loader,
    test_loader,
    criterion,
    opt_ex3,
    device,
    num_epochs=EX3_EPOCHS,
    scheduler=sched_ex3,
    model_name="Ex3_ResNet50"
)

# Results and Visualization
plot_history(hist_ex3, "Exercise 3: ResNet50 Fine-Tuning")
print("Final LRs:", [g["lr"] for g in opt_ex3.param_groups])

# --- Save and load (checkpoint) ---
ckpt_path = "ex3_resnet50.pth"
torch.save(ex3_model.state_dict(), ckpt_path)

# Verification
_, acc_before = evaluate_model(ex3_model, test_loader, criterion, device)

# Load into a fresh net architecture
loaded_model = setup_resnet50_finetune(10).to(device)
loaded_model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))

_, acc_after = evaluate_model(loaded_model, test_loader, criterion, device)

print(f"Accuracy before saving: {acc_before:.4f}")
print(f"Accuracy after loading:  {acc_after:.4f}")
print("Checkpoint verification successful!" if abs(acc_before - acc_after) < 1e-5 else "Warning: Accuracy mismatch!")

# === YOUR CODE ENDS HERE ===


### Exercise 3 — Question 1 (≥100 words)

**Prompt:** Explain **fine-tuning** here: why we unfreeze **`layer4`** but keep **`layer1–3`** frozen, and why the **learning rate** for `layer4` is usually **smaller** than for the classifier head. What would likely happen if you used one **large** learning rate for the whole network?

**Answer:**
Fine-tuning is a more aggressive form of transfer learning where we update not just the classifier head, but also some of the deeper layers of the pre-trained backbone. In Exercise 3, we unfreeze `layer4` of ResNet50 while keeping `layers 1–3` frozen. We do this because the earlier layers contain very general features (like edges) that do not need to change, whereas `layer4` contains more complex, abstract features that might benefit from being slightly adapted to better fit the specific visual patterns of the CIFAR-10 dataset.

We use a smaller learning rate for `layer4` (1e-4) compared to the classifier head (1e-3) to avoid "catastrophic forgetting." The weights in `layer4` are already quite good, having been trained on ImageNet; a large learning rate would likely destroy these useful pre-trained features by making massive, erratic updates. If we were to use one large learning rate for the whole network, the net would likely lose the valuable knowledge it gained from ImageNet, essentially reverting to training from scratch, which could lead to divergence or severe overfitting. By using differential learning rates, we gently nudge the backbone layers to adapt while allowing the new classifier head to learn the mappings to the 10 target classes more rapidly.

Word count: 218

---
<a id="conclusion--submission"></a>
# Wrap-up & Submission

## Summary of Key Learnings
- Transfer learning adapts ImageNet features to CIFAR-10 efficiently.
- **Feature extraction** trains only the head; **fine-tuning** updates top convolutional blocks with care.
- Matching **normalization**, **image size**, and **learning rates** to the pre-trained net is critical.

### Utility: Clean Notebook Metadata for GitHub
Run the cell below if you encounter the "Invalid Notebook: 'state' key is missing" error when uploading to GitHub. This script creates a 'clean' version of a notebook file by removing widget metadata.

In [ ]:
import nbformat
import os

def clean_notebook_metadata(input_filename, output_filename):
    if not os.path.exists(input_filename):
        print(f"Error: {input_filename} not found.")
        return

    # Load the notebook
    with open(input_filename, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    # Remove the problematic widgets metadata
    if 'widgets' in nb.metadata:
        del nb.metadata['widgets']
        print("Removed 'widgets' from metadata.")
    else:
        print("No 'widgets' metadata found.")

    # Save the cleaned version
    with open(output_filename, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)
    print(f"Successfully saved cleaned notebook to: {output_filename}")

# Example usage:
# 1. Manually save your notebook in Colab (File -> Save)
# 2. Find the path (usually in /content/)
# clean_notebook_metadata('your_notebook.ipynb', 'github_ready_notebook.ipynb')

### 🚀 Automated GitHub-Ready Export
1. **Press Ctrl+S** (or File > Save) right now.
2. Run the cell below.
3. Click the download link provided to get your cleaned `.ipynb` file.

In [ ]:
import nbformat
from google.colab import files
import os
import glob

# 1. Find the current notebook file
# In Colab, the notebook is usually in /content/ but filenames can vary
notebooks = glob.glob("*.ipynb")
if not notebooks:
    print("No .ipynb files found in /content/. Please ensure you have saved the notebook.")
else:
    # Assuming the largest/latest modified is the current one if multiple exist
    input_file = max(notebooks, key=os.path.getmtime)
    output_file = "github_ready_" + input_file

    print(f"Cleaning: {input_file}")

    with open(input_file, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    # 2. Aggressively remove widgets and cell metadata that causes issues
    if 'widgets' in nb.metadata:
        del nb.metadata['widgets']
        print("✅ Removed 'widgets' from notebook metadata.")

    # Also check individual cells for metadata that might confuse GitHub
    for cell in nb.cells:
        if 'metadata' in cell and 'widgets' in cell.metadata:
            del cell.metadata['widgets']

    # 3. Save and Download
    with open(output_file, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)

    print(f"Done! Downloading {output_file}...")
    files.download(output_file)

### 🛠️ Final Aggressive Metadata Stripper
This version removes ALL non-essential metadata that could cause GitHub rendering errors.

In [ ]:
import nbformat
from google.colab import files
import os
import glob

# Find latest saved notebook
notebooks = glob.glob("*.ipynb")
if not notebooks:
    print("No .ipynb files found. Please manually Save (Ctrl+S) and try again.")
else:
    input_file = max(notebooks, key=os.path.getmtime)
    output_file = "FINAL_CLEAN_EXPORT.ipynb"

    print(f"Processing: {input_file}")
    with open(input_file, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    # 1. Strip top-level metadata
    keys_to_keep = ['kernelspec', 'language_info']
    nb.metadata = {k: v for k, v in nb.metadata.items() if k in keys_to_keep}

    # 2. Strip every cell's metadata and outputs of any widget trace
    for cell in nb.cells:
        # Clear cell metadata
        cell.metadata = {}
        # Check outputs for widgets
        if 'outputs' in cell:
            new_outputs = []
            for out in cell.outputs:
                if 'data' in out:
                    # Remove interactive widget data types
                    out.data = {k: v for k, v in out.data.items() if 'widget' not in k}
                new_outputs.append(out)
            cell.outputs = new_outputs

    # 3. Write and Download
    with open(output_file, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)

    print(f"✅ Success! Please upload {output_file} to GitHub.")
    files.download(output_file)